# 05 — Modelos baseline · Versión final

**Taller de Análisis de Datos I · TP3 · 17/11/2026**

Comparamos cuatro correcciones sencillas para temperatura a 12 y 24 horas. Todos los parámetros se aprenden exclusivamente en entrenamiento purgado; se evalúan sobre la misma validación que 04. No se presuponen resultados ni un ganador.

| Modelo | Corrección sumada al pronóstico |
|---|---|
| B0 | Cero: pronóstico original |
| B1 | Error medio global del entrenamiento seguro |
| B2 | Error medio del entrenamiento seguro para cada horizonte |
| B3 | Error medio por estación suavizado hacia B1, con fuerza fija 100 |

Error = observado − pronosticado. Residual = observado − (pronosticado + corrección). Un sesgo positivo indica subestimación de temperatura.

## Ejecución

Guardar en `TP3/notebooks/` o `TP3/`, reiniciar el kernel y ejecutar todas las celdas. Dependencias: Python 3.10+, pandas 2+, NumPy, matplotlib e IPython. No necesita `tp3_config.py`, scikit-learn ni el kernel del 04.

La **Celda 1 — Code** busca la última ejecución completa del 04 por fecha UTC del identificador `YYYYMMDDTHHMMSSZ_*`; el identificador completo resuelve empates. Ignora ejecuciones sin manifiesto o con estado distinto de `completo`. Un manifiesto ilegible, contrato incompatible, archivo ausente o hash incorrecto detiene el proceso: no activa el fallback ni selecciona silenciosamente otro artefacto.

Si no hay una ejecución completa, usa los CSV con el mismo inventario y reglas del 04: 61 archivos = 48 históricos + 5 excluidos + 8 de test. Solo lee el contenido de los 48 históricos. Los otros 13 se inventarían, no se usan para ajustar ni evaluar.

**Contrato real del 04:** `features_historicas_tp3.pkl.gz` contiene el histórico completo, sin columna `particion`. Por eso este notebook reconstruye y valida las particiones, en vez de asumir una columna inexistente. Si el artefacto incluye esa columna, se comprueba su concordancia. Solo cargar pickle de una ejecución propia o de confianza; el hash verifica integridad, no autoría.

Salidas únicas: `TP3/resultados_dashboard_05/<id_ejecucion>/`. Se crea una carpeta nueva en cada ejecución; `manifest.json` se publica al final. Su ausencia indica una ejecución incompleta. No se modifica ningún archivo del 04.

**Validación de esta entrega:** ejecución completa con 61 CSV sintéticos por ambas rutas (fallback y artefacto 04), igualdad de particiones y métricas, respaldo de B3 para estaciones nuevas, purga estricta, bloqueo de artefactos alterados, hashes de salidas, reconciliación tras serialización, mejoras negativas y MAE B0 igual a cero. Falta ejecutar con los CSV reales; este archivo se entrega sin outputs ni cifras precargadas.


## Celda 1 — Markdown · Datos, trazabilidad y purga causal

| Partición | Fechas objetivo UTC |
|---|---|
| Entrenamiento candidato | 10/09–17/10/2026 |
| Validación | 18/10–01/11/2026 |
| Exclusión previa por calidad | 22–26/10/2026 |
| Test aislado, no evaluado aquí | 02–09/11/2026 |

Emisión efectiva = máximo de generación y recolección del pronóstico. Disponibilidad del error = máximo de emisión efectiva y recolección de la observación + 2 horas. Una fila candidata solo pasa a `train` si su disponibilidad es **estrictamente anterior a la primera emisión de validación**. Las demás quedan identificadas como `purgado`.

El margen de dos horas es una hipótesis compartida con 04. Se validan fechas, intervalos, claves y horizontes; no se fuerzan los conteos de ejecuciones antiguas. Cuando la fuente es 04, se verifica el contrato, el SHA-256 del histórico, sus conteos y el corte efectivo registrado.


In [ ]:
# Celda 1 — Code | Carga autónoma, fuente 04 o fallback y particiones idénticas
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json, hashlib, platform, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

VERSION = '05_baselines_final_v1'
MARGEN_H = 2
FUERZA_B3 = 100
INICIO_HIST = pd.Timestamp('2026-09-10', tz='UTC')
CORTE_VAL = pd.Timestamp('2026-10-18', tz='UTC')
INICIO_TEST = pd.Timestamp('2026-11-02', tz='UTC')
EXCLUIDOS = set(pd.date_range('2026-10-22', '2026-10-26').strftime('%Y-%m-%d'))
FECHAS_HIST = set(pd.date_range('2026-09-10','2026-11-01').strftime('%Y-%m-%d')) - EXCLUIDOS
CLAVE = ['station_id', 'target_time_utc', 'horizon_target_hours']
FECHAS = ['snapshot_time_utc','forecast_generated_at_utc','collection_time_utc',
          'target_time_utc','observation_time_utc','observation_collection_hour_utc',
          'forecast_start_utc','forecast_end_utc']
NUMERICAS = ['latitude','longitude','horizon_target_hours','temperature_forecast_c',
             'temperature_observed_c','relative_humidity_forecast_pct','dewpoint_forecast_c',
             'wind_speed_forecast_kmh','precip_probability_forecast_pct']
COLUMNAS = ['station_id','state','match_type','wind_direction_forecast'] + FECHAS + NUMERICAS

def exigir(condicion, mensaje):
    if not bool(condicion):
        raise ValueError(mensaje)

def sha256_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open('rb') as f:
        for bloque in iter(lambda: f.read(1024*1024), b''):
            h.update(bloque)
    return h.hexdigest()

def huella_claves(df):
    # Convención independiente del orden de entrada; station_id siempre es texto.
    d = df[CLAVE].sort_values(CLAVE, kind='stable').copy()
    d['target_time_utc'] = d.target_time_utc.dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    return hashlib.sha256(d.to_csv(index=False, lineterminator='\n').encode('utf-8')).hexdigest()

def ultima_completa(raiz):
    candidatas = []
    for carpeta in sorted((raiz/'resultados_dashboard_04').glob('*')):
        p = carpeta/'manifest.json'
        if not carpeta.is_dir() or not p.is_file():
            continue
        try:
            m = json.loads(p.read_text(encoding='utf-8'))
        except (ValueError, OSError) as e:
            raise ValueError(f'Manifiesto 04 ilegible: {p}. Revisar antes de continuar.') from e
        exigir(isinstance(m, dict), f'Manifiesto 04 inválido: {p}')
        if m.get('estado') != 'completo':
            continue
        identificador = m.get('id_ejecucion', '')
        exigir(identificador == carpeta.name and re.fullmatch(r'[0-9]{8}T[0-9]{6}Z_[A-Za-z0-9]+', identificador),
               f'Identificador 04 no reconocido: {p}')
        fecha = datetime.strptime(identificador.split('_')[0], '%Y%m%dT%H%M%SZ').replace(tzinfo=timezone.utc)
        candidatas.append((fecha, identificador, carpeta, m))
    return max(candidatas, key=lambda x:(x[0],x[1]))[2:] if candidatas else (None, None)

def archivo_verificado(carpeta, manifest, nombre):
    entradas = [a for a in manifest.get('archivos', []) if a.get('nombre') == nombre]
    exigir(len(entradas) == 1, f'{nombre}: falta entrada única en manifiesto 04')
    ruta, registro = carpeta/nombre, entradas[0]
    exigir(ruta.is_file(), f'Falta artefacto 04: {ruta}')
    exigir(ruta.stat().st_size == registro.get('bytes'), f'Tamaño alterado: {ruta}')
    exigir(sha256_archivo(ruta) == registro.get('sha256'), f'SHA-256 incorrecto: {ruta}')
    return ruta

def normalizar_y_controlar(df):
    exigir(set(COLUMNAS) <= set(df.columns), f'Faltan columnas: {sorted(set(COLUMNAS)-set(df.columns))}')
    r = df.copy()
    r['station_id'] = r.station_id.astype('string')
    for c in NUMERICAS:
        r[c] = pd.to_numeric(r[c], errors='coerce')
    for c in FECHAS:
        r[c] = pd.to_datetime(r[c], utc=True, errors='coerce')
    exigir(r[FECHAS].notna().all().all(), 'Marcas temporales nulas/inválidas')
    exigir((r.station_id.notna() & r.station_id.str.strip().ne('')).all(), 'Identificador de estación faltante')
    exigir((r.horizon_target_hours.isin([12,24]) & r.match_type.eq('interval')
            & np.isfinite(r.temperature_forecast_c) & np.isfinite(r.temperature_observed_c)).all(),
           'El histórico contiene filas no elegibles')
    r['horizon_target_hours'] = r.horizon_target_hours.astype('int64')
    fecha = r.target_time_utc.dt.strftime('%Y-%m-%d')
    exigir(fecha.isin(FECHAS_HIST).all(), 'Test, fechas excluidas o fuera del histórico en la entrada')
    if 'archivo_origen' in r:
        exigir(r.archivo_origen.eq('forecast_observation_' + fecha.str.replace('-','_') + '.csv').all(),
               'Fecha objetivo no coincide con archivo de origen')
    exigir(np.isclose((r.target_time_utc-r.snapshot_time_utc).dt.total_seconds()/3600,
                       r.horizon_target_hours, atol=.001, rtol=0).all(), 'Horizonte inconsistente')
    exigir((r.observation_time_utc.ge(r.forecast_start_utc)
            & r.observation_time_utc.lt(r.forecast_end_utc)).all(), 'Observación fuera del intervalo')
    exigir(r.observation_collection_hour_utc.ge(r.observation_time_utc).all(), 'Recolección anterior a observación')
    exigir((r.latitude.between(-90,90) & r.longitude.between(-180,180)).all(), 'Coordenadas inválidas')
    emision = r[['collection_time_utc','forecast_generated_at_utc']].max(axis=1)
    disponible = pd.concat([emision, r.observation_collection_hour_utc+pd.Timedelta(hours=MARGEN_H)],axis=1).max(axis=1)
    exigir(emision.lt(r.target_time_utc).all(), 'Emisión no anterior al objetivo')
    for col, calculado in [('momento_emision_utc',emision), ('error_disponible_utc',disponible)]:
        if col in r:
            exigir(pd.to_datetime(r[col],utc=True,errors='coerce').eq(calculado).all(), f'{col} difiere de la regla 04')
        r[col] = calculado
    error = r.temperature_observed_c-r.temperature_forecast_c
    if 'error_c' in r:
        exigir(np.allclose(pd.to_numeric(r.error_c,errors='coerce'),error,atol=1e-12,rtol=0), 'error_c inconsistente')
    r['error_c'], r['fecha'] = error, fecha
    r[NUMERICAS] = r[NUMERICAS].replace([np.inf,-np.inf],np.nan)
    conocidas = set('N NNE NE ENE E ESE SE SSE S SSW SW WSW W WNW NW NNW'.split())
    direccion = r.wind_direction_forecast.astype('string').str.strip().str.upper()
    exigir(~(r.wind_speed_forecast_kmh.gt(0) & ~direccion.isin(conocidas)).any(), 'Dirección desconocida con viento positivo')
    exigir(not r.duplicated(CLAVE).any(), 'Clave estación/objetivo/horizonte duplicada')
    return r

def cargar_csv(raiz):
    dataset = raiz/'dataset'
    fechas = pd.date_range('2026-09-10','2026-11-09')
    esperado = {f'forecast_observation_{d:%Y_%m_%d}.csv':str(d.date()) for d in fechas}
    encontrados = {p.name:p for p in dataset.glob('forecast_observation_*.csv')}
    exigir(set(encontrados) == set(esperado),
           f'Inventario incompleto/inesperado. Faltan: {sorted(set(esperado)-set(encontrados))}; '
           f'sobran: {sorted(set(encontrados)-set(esperado))}')
    fuentes, partes, calidad = [], [], []
    for nombre, fecha in esperado.items():
        p = encontrados[nombre]
        uso = 'historico' if fecha in FECHAS_HIST else ('excluido' if fecha in EXCLUIDOS else 'test')
        fuentes.append({'archivo':nombre,'fecha':fecha,'uso':uso,'bytes':p.stat().st_size,
                        'mtime_ns':p.stat().st_mtime_ns})
        if uso != 'historico':
            continue  # No leer contenido de test ni de días excluidos.
        fuentes[-1]['sha256'] = sha256_archivo(p)
        leidas = retenidas = 0
        for b in pd.read_csv(p,usecols=COLUMNAS,dtype={'station_id':'string','state':'string'},chunksize=100_000):
            leidas += len(b)
            for col in NUMERICAS:
                b[col] = pd.to_numeric(b[col],errors='coerce')
            filtro = (b.horizon_target_hours.isin([12,24]) & b.match_type.eq('interval')
                      & np.isfinite(b.temperature_forecast_c) & np.isfinite(b.temperature_observed_c))
            b = b.loc[filtro].copy()
            b['archivo_origen'] = nombre
            retenidas += len(b)
            if len(b):
                partes.append(b)
        exigir(retenidas > 0, f'{nombre}: sin datos elegibles')
        calidad.append({'archivo':nombre,'filas_leidas':leidas,'filas_elegibles':retenidas,
                        'filas_no_elegibles':leidas-retenidas})
        exigir(sha256_archivo(p) == fuentes[-1]['sha256'], f'CSV modificado durante la lectura: {p}')
    exigir(len([x for x in fuentes if x['uso']=='historico'])==48, 'No son 48 históricos')
    exigir(len([x for x in fuentes if x['uso']=='test'])==8, 'No son 8 archivos test aislados')
    return pd.concat(partes,ignore_index=True), pd.DataFrame(fuentes), pd.DataFrame(calidad)

raiz_proyecto = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p/'resultados_dashboard_04').is_dir() or (p/'dataset').is_dir()), None)
exigir(raiz_proyecto is not None, 'Abrir desde TP3/ o TP3/notebooks/: no se encuentra dataset/ ni resultados_dashboard_04/.')
carpeta_04, manifest_04 = ultima_completa(raiz_proyecto)
calidad_carga = pd.DataFrame()
if carpeta_04 is not None:
    protocolo = manifest_04.get('protocolo', {})
    exigir(protocolo.get('version') == '04_dashboard_v1', 'Versión 04 no soportada: revisar contrato antes de cargar')
    exigir(protocolo.get('margen_observacion_h') == MARGEN_H, 'Margen temporal 04 distinto de 2 horas')
    exigir(pd.Timestamp(protocolo.get('corte_validacion_objetivo_utc')) == CORTE_VAL, 'Corte de validación distinto')
    exigir(pd.Timestamp(protocolo.get('inicio_test_objetivo_utc')) == INICIO_TEST, 'Inicio de test distinto')
    exigir(set(protocolo.get('excluidos',[])) == EXCLUIDOS, 'Fechas excluidas distintas')
    ruta_features = archivo_verificado(carpeta_04,manifest_04,'features_historicas_tp3.pkl.gz')
    df_baseline = normalizar_y_controlar(pd.read_pickle(ruta_features,compression='gzip'))
    inventario_fuente = pd.DataFrame(manifest_04.get('fuentes',[]))
    exigir({'archivo','fecha','uso'} <= set(inventario_fuente.columns), 'Inventario 04 incompleto')
    for uso, fechas in [('historico',FECHAS_HIST),('test',set(pd.date_range('2026-11-02','2026-11-09').strftime('%Y-%m-%d'))),('excluido',EXCLUIDOS)]:
        inv = inventario_fuente.loc[inventario_fuente.uso.eq(uso)]
        exigir(len(inv)==len(fechas) and set(inv.fecha)==fechas, f'Inventario 04 inválido para {uso}')
    exigir(len(inventario_fuente)==61 and inventario_fuente.archivo.is_unique, 'Inventario 04 duplicado/incompleto')
    fuente = {'modo':'artefacto_04','id_ejecucion_04':manifest_04['id_ejecucion'],
              'ruta_relativa':str(carpeta_04.relative_to(raiz_proyecto)),
              'manifest_sha256':sha256_archivo(carpeta_04/'manifest.json'),
              'features_sha256':sha256_archivo(ruta_features)}
else:
    crudo, inventario_fuente, calidad_carga = cargar_csv(raiz_proyecto)
    df_baseline = normalizar_y_controlar(crudo)
    del crudo
    fuente = {'modo':'csv_fallback','id_ejecucion_04':None,
              'motivo':'No existe una ejecución completa del 04'}

exigir(set(df_baseline.fecha)==FECHAS_HIST, 'Faltan fechas históricas en los datos')
cobertura = pd.crosstab(df_baseline.fecha,df_baseline.horizon_target_hours).reindex(columns=[12,24],fill_value=0)
exigir((cobertura>0).all().all(), 'Falta algún horizonte en una fecha histórica')
mascara_val = df_baseline.target_time_utc.ge(CORTE_VAL)
primera_emision_validacion = df_baseline.loc[mascara_val,'momento_emision_utc'].min()
mascara_train = ~mascara_val & df_baseline.error_disponible_utc.lt(primera_emision_validacion)
particion = pd.Series(np.where(mascara_val,'validacion',np.where(mascara_train,'train','purgado')),index=df_baseline.index)
if 'particion' in df_baseline:
    exigir(df_baseline.particion.eq(particion).all(), 'Particiones preexistentes difieren de la purga causal')
df_baseline['particion'] = particion
df_train_baseline = df_baseline.loc[mascara_train].sort_values(CLAVE,kind='stable').reset_index(drop=True)
df_val_baseline = df_baseline.loc[mascara_val].sort_values(CLAVE,kind='stable').reset_index(drop=True)
exigir(len(df_train_baseline)>0 and len(df_val_baseline)>0, 'Partición vacía')
exigir(set(df_train_baseline.horizon_target_hours)=={12,24}, 'Falta horizonte en train')
exigir(df_train_baseline.error_disponible_utc.lt(primera_emision_validacion).all(), 'Purga causal incumplida')
if manifest_04 is not None:
    for nombre,n in [('historico',len(df_baseline)),('train',len(df_train_baseline)),('validacion',len(df_val_baseline))]:
        exigir(manifest_04.get('filas',{}).get(nombre)==n, f'Conteo {nombre} difiere del 04')
    exigir(pd.Timestamp(manifest_04['protocolo']['primera_emision_validacion_utc']) == primera_emision_validacion,
           'Primera emisión de validación difiere del 04')

huellas = {'train':huella_claves(df_train_baseline),'validacion':huella_claves(df_val_baseline)}
resumen_particiones = df_baseline.groupby('particion').agg(
    registros=('station_id','size'), estaciones=('station_id','nunique'),
    objetivo_desde=('target_time_utc','min'), objetivo_hasta=('target_time_utc','max'),
    disponibilidad_maxima=('error_disponible_utc','max')).reset_index()
fecha_inicio_ejecucion = datetime.now(timezone.utc)
id_ejecucion = fecha_inicio_ejecucion.strftime('%Y%m%dT%H%M%SZ')+'_'+uuid4().hex[:8]
SALIDA = raiz_proyecto/'resultados_dashboard_05'/id_ejecucion
SALIDA.mkdir(parents=True,exist_ok=False)
print('Fuente:',fuente)
print('Primera emisión de validación:',primera_emision_validacion)
print('Salidas:',SALIDA)
display(resumen_particiones)


## Celda 2 — Markdown · B0 y B1: referencia y sesgo global

B0 conserva el pronóstico original. B1 suma el error medio aprendido en `train`. Si ese promedio es positivo, eleva la temperatura pronosticada; si es negativo, la reduce.

Analizaremos el sesgo aprendido y el rendimiento global y por horizonte. Reducir el sesgo medio no garantiza reducir el MAE: B1 ajusta una media, no la mediana que minimizaría el error absoluto para una corrección constante. La validación solo se utiliza para medir rendimiento.


In [ ]:
# Celda 2 — Code | B0, B1 y evaluación común
NOMBRES = {'b0':'B0 - Original','b1':'B1 - Sesgo global',
           'b2':'B2 - Sesgo por horizonte','b3':'B3 - Sesgo por estación'}
error_train = df_train_baseline.error_c.to_numpy(float)
error_val = df_val_baseline.error_c.to_numpy(float)
horizontes_val = df_val_baseline.horizon_target_hours.to_numpy()
sesgo_global_train = float(error_train.mean())
correcciones = {'b0':np.zeros(len(error_val)), 'b1':np.full(len(error_val),sesgo_global_train)}

def evaluar_baseline(codigo,real,correccion,horizonte):
    exigir(len(real)>0 and len(real)==len(correccion), 'Vectores incompatibles/vacíos')
    residual = np.asarray(real,float)-np.asarray(correccion,float)
    exigir(np.isfinite(residual).all(), 'Residual no finito')
    return {'codigo':codigo,'modelo':NOMBRES[codigo],'horizonte_h':horizonte,'registros':len(real),
            'MAE_c':float(np.abs(residual).mean()),'RMSE_c':float(np.sqrt(np.mean(residual**2))),
            'sesgo_c':float(residual.mean())}

def evaluar_todos(predicciones):
    filas = []
    for h in ['Global',12,24]:
        mascara = np.ones(len(error_val),dtype=bool) if h=='Global' else horizontes_val==h
        for codigo,corr in predicciones.items():
            filas.append(evaluar_baseline(codigo,error_val[mascara],np.asarray(corr)[mascara],h))
    tabla = pd.DataFrame(filas)
    referencia = tabla.loc[tabla.codigo.eq('b0')].set_index('horizonte_h').MAE_c
    tabla['MAE_original_c'] = tabla.horizonte_h.map(referencia)
    tabla['mejora_MAE_pct'] = 100*(1-tabla.MAE_c/tabla.MAE_original_c.where(tabla.MAE_original_c.ne(0)))
    return tabla

print(f'Sesgo global aprendido solo en train: {sesgo_global_train:+.4f} °C')
display(evaluar_todos(correcciones).round(4))


## Celda 3 — Markdown · B2: sesgo diferenciado por horizonte

B2 aprende **dos correcciones**, una para 12 horas y otra para 24 horas, utilizando únicamente sus registros de entrenamiento seguro. Cada fila de validación recibe la corrección correspondiente a su horizonte.

La tabla permitirá comparar la dirección y magnitud de ambos sesgos, la cantidad de registros que los respaldan y si esta diferenciación mejora el MAE respecto a B0 y B1. No se ajustan estos valores con los errores de validación. Si falta un horizonte en entrenamiento, el notebook se detiene para no inventar una estimación.


In [ ]:
# Celda 3 — Code | B2
sesgo_por_horizonte = df_train_baseline.groupby('horizon_target_hours').error_c.agg(
    sesgo_aprendido_c='mean',registros_train='count')
exigir(set(sesgo_por_horizonte.index)=={12,24}, 'B2 requiere ambos horizontes en train')
correcciones['b2'] = df_val_baseline.horizon_target_hours.map(
    sesgo_por_horizonte.sesgo_aprendido_c).to_numpy(float)
exigir(np.isfinite(correcciones['b2']).all(), 'Corrección B2 faltante')
display(sesgo_por_horizonte.round(4))
display(evaluar_todos(correcciones).round(4))


## Celda 4 — Markdown · B3: sesgo suavizado por estación

B3 aprende una corrección por estación, agrupando sus dos horizontes. No es un modelo estación × horizonte. La fuerza 100 se fija antes de evaluar:

$$c_s = \frac{n_s\bar e_s + 100\bar e_{train}}{n_s+100}.$$

Con pocos registros, la corrección se acerca al sesgo global. Con muchos, pesa más la media de la estación. Una estación ausente en entrenamiento recibe exactamente B1. Se informan las estaciones y las filas que utilizaron ese respaldo. B3 puede mejorar o empeorar respecto a los otros baselines.


In [ ]:
# Celda 4 — Code | B3 con fuerza fija y respaldo global
estadisticas_estaciones = df_train_baseline.groupby('station_id').error_c.agg(
    sesgo_estacion_c='mean',registros_train='count')
estadisticas_estaciones['peso_estacion'] = estadisticas_estaciones.registros_train/(estadisticas_estaciones.registros_train+FUERZA_B3)
estadisticas_estaciones['sesgo_B3_c'] = (
    estadisticas_estaciones.registros_train*estadisticas_estaciones.sesgo_estacion_c
    + FUERZA_B3*sesgo_global_train)/(estadisticas_estaciones.registros_train+FUERZA_B3)
respaldo_b3 = ~df_val_baseline.station_id.isin(estadisticas_estaciones.index)
correcciones['b3'] = df_val_baseline.station_id.map(estadisticas_estaciones.sesgo_B3_c).fillna(sesgo_global_train).to_numpy(float)
exigir(np.isfinite(correcciones['b3']).all(), 'Corrección B3 no finita')
exigir(np.allclose(correcciones['b3'][respaldo_b3.to_numpy()],sesgo_global_train,atol=1e-12,rtol=0),
       'B3 no respeta el respaldo global')
print('Estaciones aprendidas:',len(estadisticas_estaciones))
print('Estaciones de validación sin entrenamiento:',df_val_baseline.loc[respaldo_b3,'station_id'].nunique())
print('Filas de validación con respaldo global:',int(respaldo_b3.sum()))
display(estadisticas_estaciones.reset_index().head(20).round(4))
display(evaluar_todos(correcciones).round(4))


## Celda 5 — Markdown · Comparación objetiva y gráfico

Se muestran MAE, RMSE y sesgo para cada baseline sobre las mismas filas. La reducción porcentual del MAE se calcula frente a B0; un valor negativo indica empeoramiento. Cuando B0 tiene MAE cero, el porcentaje no está definido y se indica como N/D, sin sustituirlo por cero.

El ranking principal usa la media del MAE de 12 y 24 horas, el mismo criterio del 04. También se informa el MAE global ponderado por filas; los dos rankings pueden diferir. No se exige que B3 gane ni que algún modelo mejore. Un empate numérico se declara como tal.


In [ ]:
# Celda 5 — Code | Métricas, reconciliación B0 con 04 y gráfico dinámico
resultados = evaluar_todos(correcciones)
df_baselines_global_03 = resultados.loc[resultados.horizonte_h.eq('Global')].reset_index(drop=True)
df_baselines_03 = resultados.loc[~resultados.horizonte_h.eq('Global')].reset_index(drop=True)
exigir(len(df_baselines_global_03)==4 and len(df_baselines_03)==8, 'Resultados incompletos')

# Si se consumió 04, contrastar Original con B0, sobre validación, antes de exportar.
if manifest_04 is not None:
    for nombre, esperado in [('metricas_globales.csv',df_baselines_global_03),
                             ('metricas_horizonte.csv',df_baselines_03)]:
        referencia = pd.read_csv(archivo_verificado(carpeta_04,manifest_04,nombre))
        referencia = referencia.loc[referencia.conjunto.eq('validacion') & referencia.modelo.eq('Original')]
        exigir(len(referencia)==(1 if nombre=='metricas_globales.csv' else 2), f'{nombre}: referencia Original ambigua')
        for _, fila in esperado.loc[esperado.codigo.eq('b0')].iterrows():
            ref = referencia if fila.horizonte_h=='Global' else referencia.loc[referencia.horizon_target_hours.eq(fila.horizonte_h)]
            exigir(len(ref)==1 and int(ref.iloc[0]['n'])==fila.registros, 'B0: conteos difieren del 04')
            for metrica in ['MAE_c','RMSE_c','sesgo_c']:
                exigir(np.isclose(fila[metrica],ref.iloc[0][metrica],atol=1e-9,rtol=0), f'B0 {metrica} difiere del 04')

ranking = df_baselines_03.groupby(['codigo','modelo'],as_index=False).MAE_c.mean().rename(columns={'MAE_c':'MAE_macro_horizontes_c'})
ranking = ranking.sort_values(['MAE_macro_horizontes_c','codigo'],kind='stable').reset_index(drop=True)
mejor_valor = ranking.MAE_macro_horizontes_c.min()
ganadores = ranking.loc[np.isclose(ranking.MAE_macro_horizontes_c,mejor_valor,atol=1e-12,rtol=0),'modelo'].tolist()
display(df_baselines_global_03.sort_values('MAE_c',kind='stable').round(4))
display(df_baselines_03.sort_values(['horizonte_h','MAE_c'],kind='stable').round(4))
display(ranking.round(4))
print('Menor MAE medio entre horizontes (empates incluidos):',', '.join(ganadores))

tabla_grafico = df_baselines_03.pivot(index='codigo',columns='horizonte_h',values='mejora_MAE_pct').reindex(NOMBRES)
x = np.arange(4)
fig, ax = plt.subplots(figsize=(10,5))
for offset,h in [(-.18,12),(.18,24)]:
    valores = tabla_grafico[h].to_numpy(float)
    posiciones = x+offset
    finitos = np.isfinite(valores)
    barras = ax.bar(posiciones[finitos],valores[finitos],width=.36,label=f'{h} horas')
    ax.bar_label(barras,fmt='%.2f%%',padding=3)
    for pos in posiciones[~finitos]:
        ax.annotate('N/D',(pos,0),xytext=(0,5),textcoords='offset points',ha='center')
valores_finitos = tabla_grafico.to_numpy(float)
valores_finitos = valores_finitos[np.isfinite(valores_finitos)]
limite_inf = min(0.,float(valores_finitos.min())) if len(valores_finitos) else 0.
limite_sup = max(0.,float(valores_finitos.max())) if len(valores_finitos) else 0.
margen = max(.6,.15*(limite_sup-limite_inf))
ax.set_ylim(limite_inf-margen,limite_sup+margen)
ax.set_xticks(x, ['B0 · Original','B1 · Global','B2 · Horizonte','B3 · Estación'])
ax.set(title='Baselines — Validación histórica',ylabel='Reducción del MAE frente a B0 (%)')
ax.axhline(0,color='gray',linewidth=.8)
ax.grid(axis='y',alpha=.25)
ax.set_axisbelow(True)
ax.legend()
fig.tight_layout()
fig.savefig(SALIDA/'comparacion_baselines.png',dpi=180,bbox_inches='tight')
plt.show()
plt.close(fig)


## Celda 6 — Markdown · 05-R: filas, métricas aditivas y contrato del dashboard

`baselines_validacion_por_fila.csv.gz` usa formato ancho: una fila por estación, objetivo y horizonte, con las cuatro correcciones y residuales. Se ordena explícitamente por `station_id`, `target_time_utc`, `horizon_target_hours`. **Ordenar 05 no garantiza por sí solo el orden de 04/06:** el dashboard debe unir por esa clave, validar unicidad y exigir los mismos conjuntos de claves; nunca comparar posiciones sin comprobarlas. `station_id` se lee como texto y las fechas como UTC. El 04 actual solo exporta predicciones por fila de test, por lo que no deben cruzarse con estas predicciones de validación.

Cada baseline evalúa las mismas filas; no sumar registros entre modelos. Para reagrupar filtros usar las sumas exportadas:

- MAE = suma de errores absolutos / registros.
- RMSE = raíz de (suma de errores cuadrados / registros).
- Sesgo = suma de residuales / registros.

La mejora porcentual se recalcula respecto a B0 sobre el mismo subconjunto, no se promedian porcentajes de grupos. Las tablas diaria, por estación y para filtros conservan sumas y conteos. Se verifican MAE, RMSE y sesgo contra el cálculo independiente por fila, tanto global como por horizonte, y también después de volver a leer el CSV exportado.

El archivo de particiones identifica `train`, `validacion` y `purgado` y sus disponibilidades; permite a 06 comprobar exactamente las filas. Para CV en 06, B1/B2/B3 deben volver a ajustarse dentro del train purgado de **cada fold**, sin reutilizar sesgos aprendidos con todo el histórico. Aquí no se evalúa test.


In [ ]:
# Celda 6 — Code | Exportación unificada y reconciliación
columnas_base = CLAVE + ['state','latitude','longitude','fecha','momento_emision_utc',
                        'temperature_forecast_c','temperature_observed_c']
df_base_dashboard = df_val_baseline[columnas_base].copy()
df_base_dashboard['conjunto'] = 'validacion'
df_base_dashboard['id_ejecucion_05'] = id_ejecucion
df_base_dashboard['error_real_c'] = error_val
df_base_dashboard['b3_respaldo_global_flag'] = respaldo_b3.to_numpy(dtype='int8')
for codigo,corr in correcciones.items():
    df_base_dashboard[f'correccion_{codigo}_c'] = corr
    df_base_dashboard[f'temperatura_{codigo}_c'] = df_base_dashboard.temperature_forecast_c + corr
    df_base_dashboard[f'error_residual_{codigo}_c'] = error_val-corr
df_base_dashboard = df_base_dashboard.sort_values(CLAVE,kind='stable').reset_index(drop=True)
exigir(not df_base_dashboard.duplicated(CLAVE).any(), 'Claves de dashboard duplicadas')
exigir(huella_claves(df_base_dashboard)==huellas['validacion'], 'Dashboard perdió/alteró claves de validación')

def reconciliar_filas(df):
    exigir(len(df)==len(df_val_baseline), 'Exportación perdió filas')
    for codigo in NOMBRES:
        residual = df[f'error_residual_{codigo}_c'].to_numpy(float)
        exigir(np.isfinite(residual).all(), 'Residual exportado no finito')
        for h in ['Global',12,24]:
            mascara = np.ones(len(df),dtype=bool) if h=='Global' else df.horizon_target_hours.eq(h).to_numpy()
            ref = resultados.loc[resultados.codigo.eq(codigo) & resultados.horizonte_h.eq(h)]
            exigir(len(ref)==1 and int(mascara.sum())==ref.iloc[0].registros, 'Referencia de métrica ambigua')
            calculados = {'MAE_c':np.abs(residual[mascara]).mean(),
                          'RMSE_c':np.sqrt(np.mean(residual[mascara]**2)), 'sesgo_c':residual[mascara].mean()}
            for metrica,valor in calculados.items():
                exigir(np.isclose(valor,ref.iloc[0][metrica],atol=1e-9,rtol=0), f'No reconcilia {codigo}, {h}, {metrica}')

reconciliar_filas(df_base_dashboard)

def metricas_agrupadas(dimensiones):
    partes = []
    for codigo in NOMBRES:
        d = df_base_dashboard[dimensiones].copy()
        r = df_base_dashboard[f'error_residual_{codigo}_c']
        d['suma_error_absoluto_c'] = r.abs()
        d['suma_error_cuadrado_c2'] = r**2
        d['suma_residual_c'] = r
        d['registros'] = 1
        t = d.groupby(dimensiones,dropna=False,observed=True).sum(numeric_only=True).reset_index()
        t['codigo'],t['modelo'],t['conjunto'] = codigo,NOMBRES[codigo],'validacion'
        t['MAE_c'] = t.suma_error_absoluto_c/t.registros
        t['RMSE_c'] = np.sqrt(t.suma_error_cuadrado_c2/t.registros)
        t['sesgo_c'] = t.suma_residual_c/t.registros
        partes.append(t)
    m = pd.concat(partes,ignore_index=True)
    ref = m.loc[m.codigo.eq('b0'),dimensiones+['MAE_c']].rename(columns={'MAE_c':'MAE_original_c'})
    m = m.merge(ref,on=dimensiones,how='left',validate='many_to_one')
    m['mejora_MAE_pct'] = 100*(1-m.MAE_c/m.MAE_original_c.where(m.MAE_original_c.ne(0)))
    return m.sort_values(dimensiones+['codigo'],kind='stable').reset_index(drop=True)

metricas_diarias = metricas_agrupadas(['fecha','horizon_target_hours'])
metricas_estacion = metricas_agrupadas(['station_id','horizon_target_hours'])
metricas_filtros = metricas_agrupadas(['fecha','station_id','horizon_target_hours'])
for tabla in [metricas_diarias,metricas_estacion,metricas_filtros]:
    for codigo,grupo in tabla.groupby('codigo'):
        ref = df_baselines_global_03.loc[df_baselines_global_03.codigo.eq(codigo)].iloc[0]
        exigir(grupo.registros.sum()==ref.registros, 'Suma de registros agrupados incorrecta')
        exigir(np.isclose(grupo.suma_error_absoluto_c.sum()/grupo.registros.sum(),ref.MAE_c,atol=1e-9,rtol=0),
               'MAE agrupado no reconcilia')

ruta_por_fila = SALIDA/'baselines_validacion_por_fila.csv.gz'
df_base_dashboard.to_csv(ruta_por_fila,index=False,compression='gzip',encoding='utf-8')
verificacion = pd.read_csv(ruta_por_fila,dtype={'station_id':'string'})
verificacion['target_time_utc'] = pd.to_datetime(verificacion.target_time_utc,utc=True)
exigir(huella_claves(verificacion)==huellas['validacion'], 'Claves alteradas al serializar')
exigir(verificacion[CLAVE].equals(verificacion[CLAVE].sort_values(CLAVE,kind='stable').reset_index(drop=True)),
       'Orden de filas incorrecto al serializar')
reconciliar_filas(verificacion)
del verificacion

particiones_exportar = df_baseline[CLAVE+['particion','momento_emision_utc','error_disponible_utc']].sort_values(CLAVE,kind='stable')
particiones_exportar.to_csv(SALIDA/'particiones_historicas_por_fila.csv.gz',index=False,compression='gzip')
tablas = {'01_baselines_por_horizonte.csv':df_baselines_03,
          '02_baselines_globales.csv':df_baselines_global_03,
          '03_sesgos_por_estacion.csv':estadisticas_estaciones.reset_index(),
          'sesgos_por_horizonte.csv':sesgo_por_horizonte.reset_index(),
          'metricas_diarias.csv':metricas_diarias,'metricas_estacion.csv':metricas_estacion,
          'metricas_filtros.csv':metricas_filtros,'ranking_macro_horizontes.csv':ranking,
          'resumen_particiones.csv':resumen_particiones,'inventario_fuente.csv':inventario_fuente}
if not calidad_carga.empty:
    tablas['calidad_carga_csv.csv'] = calidad_carga
for nombre,tabla in tablas.items():
    tabla.to_csv(SALIDA/nombre,index=False,encoding='utf-8-sig')
parametros = {'fuerza_B3':FUERZA_B3,'sesgo_global_train_c':sesgo_global_train,
              'sesgos_por_horizonte':{str(k):float(v) for k,v in sesgo_por_horizonte.sesgo_aprendido_c.items()},
              'sesgos_B3_por_estacion':{str(k):float(v) for k,v in estadisticas_estaciones.sesgo_B3_c.items()},
              'respaldo_estacion_desconocida':'sesgo_global_train_c','ajuste':'solo_train_purgado'}
(SALIDA/'parametros_baselines.json').write_text(json.dumps(parametros,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')

descripciones = {
    'station_id':'Identificador de estación; leer como texto','target_time_utc':'Fecha/hora objetivo UTC',
    'horizon_target_hours':'Horizonte nominal: 12 o 24 horas','horizonte_h':'12, 24 o Global',
    'state':'Estado informado en el CSV','latitude':'Latitud, grados','longitude':'Longitud, grados',
    'fecha':'Fecha objetivo UTC, YYYY-MM-DD','momento_emision_utc':'Máximo de generación y captura del pronóstico',
    'temperature_forecast_c':'Pronóstico original, °C','temperature_observed_c':'Observación, °C',
    'conjunto':'Siempre validacion en estas métricas','id_ejecucion_05':'Identificador de esta ejecución',
    'error_real_c':'Observado menos pronóstico original, °C','b3_respaldo_global_flag':'1 si estación ausente en train',
    'codigo':'b0/b1/b2/b3','modelo':'Nombre legible del baseline','registros':'Filas evaluadas por modelo',
    'MAE_c':'Error absoluto medio, °C','RMSE_c':'Raíz del error cuadrático medio, °C',
    'sesgo_c':'Residual medio observado menos corregido, °C','MAE_original_c':'MAE B0 en el mismo subconjunto',
    'mejora_MAE_pct':'100*(1-MAE/MAE_B0); vacío si MAE_B0=0',
    'suma_error_absoluto_c':'Numerador aditivo del MAE','suma_error_cuadrado_c2':'Numerador aditivo del MSE',
    'suma_residual_c':'Numerador aditivo del sesgo'}
for codigo in NOMBRES:
    descripciones[f'correccion_{codigo}_c'] = f'Corrección {codigo} aprendida en train, °C'
    descripciones[f'temperatura_{codigo}_c'] = f'Pronóstico original más corrección {codigo}, °C'
    descripciones[f'error_residual_{codigo}_c'] = f'Observado menos temperatura {codigo}, °C'
campos = set(df_base_dashboard.columns)|set(metricas_filtros.columns)|set(df_baselines_03.columns)
exigir(campos<=set(descripciones), 'Falta describir columna del dashboard')
pd.DataFrame([{'columna':c,'descripcion':descripciones[c]} for c in sorted(campos)]).to_csv(
    SALIDA/'diccionario_dashboard.csv',index=False,encoding='utf-8-sig')
print('Exportaciones reconciliadas. El manifiesto se completa en la siguiente celda.')


## Celda 7 — Markdown · Conclusiones de la ejecución y cierre

La siguiente celda genera las conclusiones a partir de los resultados observados. Informa el menor MAE medio entre horizontes, el menor MAE global y la mejora o empeoramiento de cada baseline. No atribuye de antemano una ventaja a B3. Si un baseline reduce sesgo pero aumenta MAE, ambas observaciones son compatibles.

Esta comparación es una validación retrospectiva en un período acotado. La corrección por estación no demuestra causalidad geográfica ni generalización fuera del período analizado. Los registros comparten estaciones y fechas; estas tablas no constituyen por sí solas un contraste de significación estadística. Los ocho días de test permanecen sin evaluar en 05.

Las salidas solo se consideran completas cuando existe el manifiesto final, con fuente, protocolo, fechas, conteos, huellas de las claves y SHA-256 de cada archivo. No se incluye el hash del propio manifiesto para evitar una referencia circular. Para reproducir una comparación entre 04/05/06, fijar el identificador de ejecución 04 registrado y verificar las claves y la misma regla de purga.


In [ ]:
# Celda 7 — Code | Conclusiones calculadas y manifest.json al final
min_global = df_baselines_global_03.MAE_c.min()
ganadores_global = df_baselines_global_03.loc[np.isclose(df_baselines_global_03.MAE_c,min_global,atol=1e-12,rtol=0),'modelo'].tolist()
lineas = [f'Fuente: {fuente["modo"]}; ejecución 04: {fuente.get("id_ejecucion_04")}.',
          f'Entrenamiento seguro: {len(df_train_baseline):,} filas; validación: {len(df_val_baseline):,} filas.',
          'Menor MAE medio entre horizontes: '+', '.join(ganadores)+f' ({mejor_valor:.4f} °C).',
          'Menor MAE global: '+', '.join(ganadores_global)+f' ({min_global:.4f} °C).']
for fila in df_baselines_global_03.itertuples():
    cambio = f'{fila.mejora_MAE_pct:+.2f} %' if np.isfinite(fila.mejora_MAE_pct) else 'no definido (MAE B0=0)'
    lineas.append(f'{fila.modelo}: MAE {fila.MAE_c:.4f} °C, RMSE {fila.RMSE_c:.4f} °C, '
                  f'sesgo {fila.sesgo_c:+.4f} °C; reducción MAE: {cambio}.')
lineas += [f'B3 usó respaldo global en {int(respaldo_b3.sum()):,} filas.',
           'Los parámetros se ajustaron solo en entrenamiento purgado; test no se leyó ni evaluó.',
           'No se presupone mejora ni significación estadística. Son resultados de validación retrospectiva.']
texto_conclusiones = '\n'.join(lineas)
print(texto_conclusiones)
(SALIDA/'conclusiones.txt').write_text(texto_conclusiones+'\n',encoding='utf-8')

if manifest_04 is not None:
    exigir(sha256_archivo(carpeta_04/'manifest.json')==fuente['manifest_sha256'], 'Manifiesto 04 cambió durante la ejecución')
    exigir(sha256_archivo(ruta_features)==fuente['features_sha256'], 'Histórico 04 cambió durante la ejecución')
else:
    for fila in inventario_fuente.itertuples():
        p = raiz_proyecto/'dataset'/fila.archivo
        exigir(p.stat().st_size==fila.bytes and p.stat().st_mtime_ns==fila.mtime_ns, 'Fuente CSV cambió durante ejecución')
        if fila.uso=='historico':
            exigir(sha256_archivo(p)==fila.sha256, f'Contenido histórico cambió: {p}')

manifest = {'estado':'completo','version':VERSION,'id_ejecucion':id_ejecucion,
    'inicio_utc':fecha_inicio_ejecucion.isoformat(),'fin_utc':datetime.now(timezone.utc).isoformat(),
    'fuente':fuente,'versiones':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__},
    'protocolo':{'error':'observado_menos_pronosticado','margen_observacion_h':MARGEN_H,
                'corte_validacion_objetivo_utc':str(CORTE_VAL),
                'primera_emision_validacion_utc':str(primera_emision_validacion),
                'purga':'error_disponible_utc < primera_emision_validacion',
                'excluidos':sorted(EXCLUIDOS),'test_aislado':['2026-11-02','2026-11-09'],
                'fuerza_B3':FUERZA_B3,'B3_agrupacion':'station_id; ambos horizontes juntos',
                'B3_respaldo':'media global de train','ranking':'media MAE de 12 y 24 horas',
                'clave':CLAVE,'orden_exportacion':CLAVE,'ajuste':'solo train purgado'},
    'conteos':{'historico':len(df_baseline),'train_candidato':int((~mascara_val).sum()),
               'train':len(df_train_baseline),'purgado':int(df_baseline.particion.eq('purgado').sum()),
               'validacion':len(df_val_baseline),'test_leido':0,'filas_respaldo_B3':int(respaldo_b3.sum())},
    'fechas':{'historico_desde':str(df_baseline.target_time_utc.min()),
              'historico_hasta':str(df_baseline.target_time_utc.max()),
              'validacion_desde':str(df_val_baseline.target_time_utc.min()),
              'validacion_hasta':str(df_val_baseline.target_time_utc.max())},
    'huellas_claves_sha256':huellas,
    'convencion_huella':'CSV UTF-8, LF, sin índice; orden CLAVE; fecha UTC %Y-%m-%dT%H:%M:%S.%fZ',
    'reconciliacion':{'global_y_horizonte':True,'serializacion_csv':True,'tolerancia_absoluta':1e-9,
                      'tolerancia_relativa':0,'B0_con_04':manifest_04 is not None},
    'archivos':[{'nombre':p.name,'bytes':p.stat().st_size,'sha256':sha256_archivo(p)}
                for p in sorted(SALIDA.iterdir()) if p.is_file() and p.name not in ['manifest.json','manifest.tmp']]}
# Publicación atómica: antes de este paso la ejecución no está marcada como completa.
ruta_temporal = SALIDA/'manifest.tmp'
ruta_temporal.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
ruta_temporal.replace(SALIDA/'manifest.json')
print('Ejecución completa:',SALIDA)
